In [8]:
library(tidyverse)
library(readxl)
library(here)
i_am("ncrecords/import-provider-days.ipynb")

data_dir <- here("ncrecords/nctracks/institutional-days")



here() starts at /Users/sarah/Github/nc-homesdisabled


Find all of the files

In [10]:



# -----------------------------------------------------------------------------
# 1. Find the files
# -----------------------------------------------------------------------------
# Every file that has "institutional days" in its name (allowing a space,
# underscore, or hyphen between the two words), case-insensitive.
files <- list.files(
  data_dir,
  pattern    = "institutional[ _-]days",
  ignore.case = TRUE,
  full.names  = TRUE
)

print(files)



 [1] "/Users/sarah/Github/nc-homesdisabled/ncrecords/nctracks/institutional-days/SFY 2018 Service Provider Medicaid Institutional Days.csv" 
 [2] "/Users/sarah/Github/nc-homesdisabled/ncrecords/nctracks/institutional-days/SFY 2021 Service Provider Medicaid Institutional Days.xlsx"
 [3] "/Users/sarah/Github/nc-homesdisabled/ncrecords/nctracks/institutional-days/SFY 2022 Service Provider Medicaid Institutional Days.xlsx"
 [4] "/Users/sarah/Github/nc-homesdisabled/ncrecords/nctracks/institutional-days/SFY 2023 Service Provider Medicaid Institutional Days.csv" 
 [5] "/Users/sarah/Github/nc-homesdisabled/ncrecords/nctracks/institutional-days/SFY 2024 Service Provider Medicaid Institutional Days.xlsx"
 [6] "/Users/sarah/Github/nc-homesdisabled/ncrecords/nctracks/institutional-days/SFY 2025 Service Provider Medicaid Institutional Days.csv" 
 [7] "/Users/sarah/Github/nc-homesdisabled/ncrecords/nctracks/institutional-days/SFY-2019-Service-Provider-Medicaid-Institutional-Days.csv" 
 [8] "/Users/

In [ ]:


# -----------------------------------------------------------------------------
# 2. Column name standardization
# -----------------------------------------------------------------------------
# The raw column names vary a bit release to release (extra spaces, "Visits"
# vs "Visits/Days", double spaces, trailing space, etc). Map every variant
# we've seen onto one canonical name, keyed on the lower-cased/whitespace-
# squished raw name.
canonical_map <- c(
  "billing npi/atypical"              = "billing_npi",
  "billing provider name"             = "billing_provider_name",
  "billing provider dba name"         = "billing_provider_dba_name",
  "billing address"                   = "billing_address",
  "billing county code"               = "billing_county_code",
  "billing county name"               = "billing_county_name",
  "billing provider taxonomy code"    = "billing_provider_taxonomy_code",
  "billing provider type"             = "billing_provider_type",
  "billing provider classification"   = "billing_provider_classification",
  "billing provider specialization"   = "billing_provider_specialization",
  "rendering npi/atypical"            = "rendering_npi",
  "rendering provider name"           = "rendering_provider_name",
  "rendering address"                 = "rendering_address",
  "rendering county code"             = "rendering_county_code",
  "rendering county name"             = "rendering_county_name",
  "rendering taxonomy code"           = "rendering_taxonomy_code",
  "rendering provider type"           = "rendering_provider_type",
  "rendering provider classification" = "rendering_provider_classification",
  "rendering provider specialization" = "rendering_provider_specialization",
  "number of recipients"              = "number_of_recipients",
  "number of visits/days"             = "number_of_visits_or_days",
  "number of visits"                  = "number_of_visits_or_days",
  "number of procedures"              = "number_of_procedures",
  "amount paid"                       = "amount_paid"
)

# columns that should end up numeric once cleaned
numeric_cols <- c("number_of_recipients", "number_of_visits_or_days",
                   "number_of_procedures", "amount_paid")



Find first data rows for Xl files

In [51]:
get_first_row <- function ( filename,  marker) {
    if ( str_ends( filename, "csv") ) {
        peek <- read_csv( filename, 
                            col_names=FALSE,
                            n_max=60,
                            col_types=cols(.default="c"), 
                            name_repair = "unique_quiet", 
                            show_col_types = FALSE) 
    } else if (str_ends ( filename, "xls.")) {
          peek <- read_excel(filename, sheet = 1, col_names = FALSE,
                      n_max = 60, col_types = "text", .name_repair="unique_quiet")  
    }

    first_row <- peek |> 
        rowid_to_column() |> 
        mutate  ( found = if_any ( 1:5, \(x) str_starts(x, "Billing NPI")) ) |> 
        filter ( found == TRUE) 

    first_row |>  pull ( rowid) 

}



In [ ]:



get_orig_data <- function ( filename) {
  
  start_after <- get_first_row ( filename, "Billing NPI") -1    
  sfy <-  str_extract(filename, "20\\d\\d")
  ftype <- tools::file_ext( filename) 

  if ( ftype == "csv") {

    df <- read_csv( filename, skip =  start_after ,
                            col_names=TRUE,
                            col_types=cols(.default="c"), 
                            name_repair = "unique_quiet", 
                            show_col_types = FALSE, 
                            na = c("#", "$#", "")
                          )  |> 
    select ( -c(  starts_with ("...") ) )


  } else if ( ftype == "xlsx") {

    df <- read_excel( filename, 
                              skip = start_after, 
                              col_names = TRUE, 
                              col_types="text",
                               .name_repair="unique_quiet"
                              ) 

  }
 
  df |>  mutate ( fiscal_year = sfy, orig_fname = filename)

}







: [1m[33mError[39m in `map_chr()`:[22m
[1m[22m[36mℹ[39m In index: 1.
[1mCaused by error:[22m
[33m![39m Result must be length 1, not 25.

In [113]:
icf_list <- map ( all_data, \(x) x |> filter ( `Rendering Taxonomy Code`=="315P00000X"))


In [136]:
library(gt)

good_years <- 
icf_list[4:6] |>  list_rbind() |> janitor::clean_names() |> 
mutate ( across ( number_of_recipients:amount_paid , \(x) parse_number(x), .names= "num_{.col}") ) 


good_years |> 
filter  (str_starts( billing_provider_name, "RHA") ) |> 
summarize ( n = n(), amt = sum( num_amount_paid, na.rm=T) , .by= c(fiscal_year) )|> 
gt ( ) |> 
fmt_currency( amt )

Warning message:
There were 4 warnings in `mutate()`.
The first warning was:
ℹ In argument: `across(...)`.
Caused by warning:
! 989 parsing failures.
 row col expected actual
1203  -- a number      #
1204  -- a number      #
1205  -- a number      #
1206  -- a number      #
1207  -- a number      #
.... ... ........ ......
See problems(...) for more details.
ℹ Run ]8;;x-r-run:dplyr::last_dplyr_warnings()dplyr::last_dplyr_warnings()]8;; to see the 3 remaining warnings. 


fiscal_year,n,amt
2023,388,"$255,964,009.04"
2024,348,"$102,428,130.20"
2025,412,"$172,650,962.27"


In [ ]:



                   
standardize_names <- function(nm) {
  key    <- str_to_lower(str_squish(nm))
  mapped <- unname(canonical_map[key])
  unmapped <- is.na(mapped)
  # anything we don't recognize (e.g. a stray unnamed trailing column) keeps
  # a readable name instead of silently disappearing
  mapped[unmapped] <- paste0("unmapped_", str_replace_all(key[unmapped], "[^a-z0-9]+", "_"))
  mapped
}

# -----------------------------------------------------------------------------
# 3. Numeric cleanup: "#" (or "$#") means redacted -> NA; otherwise strip
#    "$" and "," and convert to numeric
# -----------------------------------------------------------------------------

# -----------------------------------------------------------------------------
# 5. Locate the header row
# -----------------------------------------------------------------------------
# Every file leads with several lines of title/run-date/footnote text before
# the real header row (line 21 in some years, 22 in others) -- rather than
# hardcode a row number per file, find the row that contains "Billing NPI".
find_header_row_csv <- function(path, marker = "Billing NPI") {
  peek <- read_lines(path, n_max = 60)
  hit  <- which(str_detect(peek, fixed(marker)))[1]
  if (is.na(hit)) stop("Could not find header row in ", path)
  hit
}

find_header_row_excel <- function(path, marker = "Billing NPI", sheet = 1) {
  peek <- read_excel(path, sheet = sheet, col_names = FALSE,
                      n_max = 60, col_types = "text")
  hit  <- which(apply(peek, 1, function(r) any(str_detect(r, fixed(marker)), na.rm = TRUE)))[1]
  if (is.na(hit)) stop("Could not find header row in ", path)
  hit
}

# -----------------------------------------------------------------------------
# 6. Read one file (csv or excel) into a standardized data frame
# -----------------------------------------------------------------------------
read_institutional_days_file <- function(path) {

  ext <- str_to_lower(tools::file_ext(path))

  if (ext == "csv") {

    header_row <- find_header_row_csv(path)
    df <- read_csv(
      path,
      skip       = header_row - 1,
      col_names  = TRUE,
      col_types  = cols(.default = col_character()),  # read raw, clean after
      na         = c("", "NA")
    )

  } else if (ext %in% c("xlsx", "xls")) {

    header_row <- find_header_row_excel(path)
    df <- read_excel(
      path,
      skip      = header_row - 1,
      col_names = TRUE,
      col_types = "text"                              # read raw, clean after
    )

  } else {
    stop("Unsupported file type: ", path)
  }

  # drop the unnamed trailing column readr/readxl invent from trailing commas
  df <- df %>% select(-matches("^\\.\\.\\.[0-9]+$"))

  # standardize names to a common schema across files
  names(df) <- standardize_names(names(df))

  # drop the footer rows: a column-totals row and/or blank rows at the very
  # end, identifiable because they have no Billing NPI
  df <- df %>% filter(!is.na(billing_npi), str_squish(billing_npi) != "")

  # coerce the numeric columns, turning "#"/"$#" (redacted small values) into NA
  present_numeric_cols <- intersect(numeric_cols, names(df))
  df <- df %>% mutate(across(all_of(present_numeric_cols), clean_numeric))

  df$source_file <- basename(path)
  df
}

# -----------------------------------------------------------------------------
# 7. Read every file, examine it, and stash it in a named list
# -----------------------------------------------------------------------------
institutional_days <- vector("list", length(files))
names(institutional_days) <- basename(files)

for (f in files) {
  message("Reading: ", basename(f))
  df <- read_institutional_days_file(f)
  institutional_days[[basename(f)]] <- df

  cat("\n----", basename(f), "----\n")
  cat("dim: ", nrow(df), "rows x", ncol(df), "cols\n")
  cat("colnames:\n")
  print(inspect_colnames(df))
  cat("coltypes:\n")
  print(inspect_coltypes(df))
}

# Each file is now available as its own data frame, e.g.:
#   institutional_days[["SFY2016_Institutional_Days.csv"]]
#   institutional_days[["SFY-2020-Service-Provider-Medicaid-Institutional-Days.xlsx"]]